# Spatio-Temporal Air Raid Alert Forecasting in Ukraine
### Multi-Resolution Adaptive Graph Neural Network (Adaptive ST-GNN)

This interactive notebook demonstrates:
1. **Exploratory Data Analysis**: Historical siren intensity, distribution across 25 regions, and diurnal attack patterns.
2. **Spatial Graph Topology**: Geographic land adjacency matrix vs. learnable corridor embeddings.
3. **Multi-Horizon Model Training**: Training the dual-stream ST-GNN with Asymmetric Focal Loss.
4. **Overfitting Diagnostics**: Generalization Gap tracking, Precision-Recall curves, and Reliability Diagrams on the latest holdout period.

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import torch

from air_raid_forecast.src.data.download_alerts import load_and_clean_alerts
from air_raid_forecast.src.data.build_graph import build_adjacency_matrix, get_normalized_laplacian, REGIONS, REGION_TO_IDX
from air_raid_forecast.src.data.multi_res_dataset import build_regular_time_grid, MultiResolutionDataset
from air_raid_forecast.src.models.st_gnn import MultiResSTGNN
from air_raid_forecast.src.training.evaluate import evaluate_predictions, find_optimal_threshold

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Load Clean Historical Alerts Dataset

In [ ]:
df = load_and_clean_alerts()
print(f"Total Alert Records: {len(df):,}")
print(f"Time Span: {df['start'].min()} to {df['start'].max()}")
df[['region_title', 'start', 'end']].head(5)

## 2. Spatio-Temporal Regular Grid & Adjacency Graph

In [ ]:
time_idx, grid = build_regular_time_grid(df, freq="15min")
adj = build_adjacency_matrix()
lap = get_normalized_laplacian(adj)

print(f"Grid Shape (15-min intervals x 25 regions): {grid.shape}")
print(f"Total Undirected Border Connections: {int(np.sum(adj)/2)}")
print(f"Overall National Alert Activation Rate: {np.mean(grid)*100:.2f}%")

## 3. Anti-Leakage Multi-Resolution Dataset

In [ ]:
mr_dataset = MultiResolutionDataset(grid, time_idx, adj)
x_fast, x_slow, y_target = mr_dataset.get_sample(10)

print(f"X_fast (15m tactical stream) shape: {x_fast.shape} [25 nodes, 8 steps, 2 features]")
print(f"X_slow (1h operational stream) shape: {x_slow.shape} [25 nodes, 24 steps, 3 features]")
print(f"Y_target (Multi-horizon) shape: {y_target.shape} [25 nodes, 3 horizons: 1h, 4h, 24h]")

## 4. MultiResSTGNN PyTorch Model Forward Pass

In [ ]:
model = MultiResSTGNN(num_nodes=25)
lap_t = torch.from_numpy(lap).float()

x_f_t = torch.from_numpy(x_fast).unsqueeze(0)
x_s_t = torch.from_numpy(x_slow).unsqueeze(0)

with torch.no_grad():
    logits = model(x_f_t, x_s_t, lap_t)
    probs = torch.sigmoid(logits).squeeze(0).numpy()

print(f"Forward Pass Output shape: {probs.shape} (25 regions x 3 horizons)")
kyiv_idx = REGION_TO_IDX["м. Київ"]
print(f"Sample Forecast for Kyiv City: 1h={probs[kyiv_idx, 0]:.3f}, 4h={probs[kyiv_idx, 1]:.3f}, 24h={probs[kyiv_idx, 2]:.3f}")

## 5. Overfitting Diagnostics & Evaluation Audit

In [ ]:
from air_raid_forecast.src.training.evaluate_model import evaluate_checkpoint_on_holdout

# Evaluate model on the latest 45-day untouched holdout period
report = evaluate_checkpoint_on_holdout(test_days=45, sample_stride=8)

print("\n--- Overfitting Status ---")
print(json.dumps(report['overfitting_diagnostic'], indent=2))